# 01. Исходный вектор и dev
Подключи Google Drive и выбери T4. В первом блоке задай признак и имя запуска.

In [ ]:
from pathlib import Path
from runpy import run_path
from subprocess import run

REPO = "https://github.com/danilov-e/lora-activation-steering.git"
RUN_NAME = "research_api_01"
TRAIT = "sycophancy"  # evil / sycophancy / hallucination
GIT_REF = "main"      # используется при создании нового запуска

SOURCE = Path("/content/lora-steering-bootstrap")
if not SOURCE.exists():
    run(["git", "clone", "--quiet", "--depth", "1", REPO, str(SOURCE)], check=True)

colab = run_path(str(SOURCE / "notebooks" / "colab.py"))
PROJECT, RUNS, COMMIT = colab["setup"](SOURCE, RUN_NAME, TRAIT, GIT_REF, resume=False)

from src import judge as j, report as r, workflow as w
from IPython.display import Image, display
import pandas as pd

## Настройки
Сетка берётся из выбранного профиля. Её можно менять до создания запуска.

In [ ]:
profile = w.read_json(PROJECT / "configs" / "research.json")[TRAIT]
CONFIG = w.default_config()
for key in ("layers", "alphas", "dev_repeats", "test_repeats"):
    CONFIG[key] = profile[key]

# CONFIG["layers"] = [17, 18, 19]
# CONFIG["alphas"] = [1, 2, 3]
# CONFIG["judge"]["model"] = "gpt-5.4-mini"

RUN = w.create_run(PROJECT, RUN_NAME, TRAIT, CONFIG, runs_dir=RUNS, git_commit=COMMIT)
DEV = RUN / "dev"
conditions = w.sweep_conditions(CONFIG)
expected = len(w.pairs(RUN, "dev")) * len(conditions) * CONFIG["dev_repeats"]
batch_size = CONFIG["judge"]["batch_size"]
print(RUN)
print(f"Dev: {expected} ответов, около {(expected + batch_size - 1) // batch_size} запросов")

## Извлечение
Посмотри нормы и split-half cosine. Слой считается с нуля.

In [ ]:
w.extract(RUN)
geometry, plot = r.extraction_report(RUN)
display(geometry.loc[geometry.layer.isin(CONFIG["layers"])].round(4))
display(Image(filename=str(plot)))

## Генерация dev
Ответы дописываются на Диск. После обрыва выполни ячейку снова.

In [ ]:
w.generate(RUN, "dev", DEV, conditions, repeats=CONFIG["dev_repeats"])

## Оценка через OpenAI
Следующая ячейка запускает платные вызовы. Ключ берётся из Colab Secrets или скрытого ввода.

In [ ]:
colab["api_key"]()
display(j.judge(DEV, TRAIT, **CONFIG["judge"]))

## Результаты и выбор
Сравни прирост признака с потерей coherence и выбери 2–3 настройки по dev.

In [ ]:
frame = r.scored(DEV, TRAIT)
summary, effects, figures = r.dev_report(frame, TRAIT, DEV)
display(summary.sort_values("target_mean", ascending=False).round(2))
display(effects.round(2))
for plot in figures:
    display(Image(filename=str(plot)))

candidates = summary[summary.condition.ne("base:baseline")].dropna(subset=["target_mean"])
if not candidates.empty:
    condition = candidates.sort_values("target_mean", ascending=False).iloc[0].condition
    display(r.examples(frame, TRAIT, condition))

In [ ]:
CHOICES = [
    # Заполни по результатам выше:
    # {"name": "c1", "layer": ..., "alpha": ...},
    # {"name": "c2", "layer": ..., "alpha": ...},
    # {"name": "c3", "layer": ..., "alpha": ...},
]
if CHOICES:
    display(w.save_choices(RUN, CHOICES))
else:
    print("Впиши две или три настройки перед запуском ноутбука 02.")